# Calculadora Agrícola IoT — ETL da base de entrada

Pipeline que transforma os arquivos horários do INMET em uma base **mensal por estação**,
já cruzada com a classificação de solo, pronta para pesar os coeficientes do Elastic Net.

**O que mudou em relação à versão anterior**

| # | Problema na versão anterior | Correção aplicada aqui |
|---|---|---|
| 1 | `groupby.mean()` em **todas** as colunas | Regra de agregação por tipo de variável |
| 2 | Precipitação virava *média horária* (0,37 mm) | Vira **total do mês** em mm |
| 3 | Radiação média não comparável entre estações | Vira **total acumulado** no mês |
| 4 | Direção do vento com média aritmética (variável circular) | **Média vetorial** (seno/cosseno) |
| 5 | Coluna fantasma `Unnamed: 19` | Removida na leitura |
| 6 | Sentinela `-9999` do INMET entrava como número | Convertida para nulo |
| 7 | Merge por cidade falhava em silêncio (tudo `NaN`) | Chave normalizada + **relatório de cobertura** |
| 8 | 4 colunas quase idênticas por família (r ≈ 1,00) | Consolidadas em variáveis agronômicas |
| 9 | Sem noção de quantas horas geraram cada média | Coluna `horas_validas` / `dias_validos` |
| 10 | Solo atribuído por estado inteiro (2 classes) | Mapa auditável por cidade + fallback explícito |

**Fluxo:** horário → diário → mensal. Agregar direto de horário para mensal impede
calcular graus-dia, veranico e amplitude térmica, que são diários por definição.


## 1. Configuração

In [ ]:
import os
import re
import glob
import unicodedata
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------- ajustes --
CAMINHO_BASE = r"C:\Users\Camassati\Desktop\TCC\arquivos para bd tcc\inmet_historico"
ESTADOS_ALVO = ["SP", "PR", "RS", "MT"]
ANO_INICIAL = 2023

# Base de graus-dia. 10 °C cobre soja e milho; trigo usa 0 °C.
TEMP_BASE_GDD = 10.0

# Um dia entra na conta se tiver pelo menos este nº de horas válidas.
MIN_HORAS_DIA = 18
# Um mês entra na conta se tiver pelo menos esta fração de dias válidos.
MIN_FRACAO_DIAS_MES = 0.80

# Chuva a partir da qual o dia conta como "dia com chuva" (padrão OMM).
LIMIAR_DIA_CHUVOSO_MM = 1.0

SAIDA_XLSX = "df_tcc_completo.xlsx"

print(f"Lendo de: {CAMINHO_BASE}")
print(f"Estados: {', '.join(ESTADOS_ALVO)} | a partir de {ANO_INICIAL}")

## 2. Leitura robusta dos arquivos do INMET

O layout dos arquivos do INMET muda de ano para ano: o cabeçalho às vezes tem 8 linhas,
às vezes 9, e os nomes das colunas variam (`TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)`
vs `Temperatura do Ar - Bulbo Seco, Horaria (°C)`). Em vez de fixar `skiprows=8`, aqui a
linha de cabeçalho é **localizada** e os nomes são normalizados para chaves internas
estáveis (`temp_ar`, `umidade`, …).

In [ ]:
# Fragmentos identificadores, EM ORDEM DE PRIORIDADE (o primeiro que casar vence).
# A ordem importa: "temperatura orvalho max" precisa ser testado antes de
# "temperatura maxima", senão a coluna errada é capturada.
#
# Atenção: NÃO usar "hora" como fragmento genérico. Metade das colunas do INMET
# contém "NA HORA ANT." no nome ("PRESSÃO ATMOSFERICA MAX.NA HORA ANT.",
# "TEMPERATURA ORVALHO MIN. NA HORA ANT."), então um fragmento "hora" captura
# todas elas de uma vez. Data e hora são identificadas separadamente, por posição.
MAPA_COLUNAS = [
    ("precipitacao total",             "precipitacao"),
    ("pressao atmosferica ao nivel",   "pressao"),
    ("radiacao global",                "radiacao"),
    ("temperatura orvalho max",        "orvalho_max"),
    ("temperatura orvalho min",        "orvalho_min"),
    ("temperatura do ponto de orvalho", "orvalho"),
    ("temperatura do ar",              "temp_ar"),
    ("temperatura maxima",             "temp_max"),
    ("temperatura minima",             "temp_min"),
    ("umidade rel. max",               "umidade_max"),
    ("umidade rel. min",               "umidade_min"),
    ("umidade relativa max",           "umidade_max"),
    ("umidade relativa min",           "umidade_min"),
    ("umidade relativa do ar",         "umidade"),
    ("vento, direcao",                 "vento_dir"),
    ("vento direcao",                  "vento_dir"),
    ("vento, rajada",                  "vento_rajada"),
    ("vento rajada",                   "vento_rajada"),
    ("vento, velocidade",              "vento_vel"),
    ("vento velocidade",               "vento_vel"),
]

# Faixas fisicamente possíveis. Fora disso é erro de sensor, não clima.
LIMITES_FISICOS = {
    "temp_ar":      (-15, 50),
    "temp_max":     (-15, 50),
    "temp_min":     (-15, 50),
    "orvalho":      (-25, 35),
    "umidade":      (0, 100),
    "umidade_max":  (0, 100),
    "umidade_min":  (0, 100),
    "pressao":      (800, 1060),
    "precipitacao": (0, 150),      # mm em UMA hora
    "radiacao":     (0, 6000),     # kJ/m² em UMA hora
    "vento_vel":    (0, 60),
    "vento_rajada": (0, 100),
    "vento_dir":    (0, 360),
}

# O INMET usa estes valores como "sem medição".
SENTINELAS = [-9999, -9999.0, "-9999", "null", "NULL", ""]


def normalizar(texto):
    """Remove acento, baixa a caixa e colapsa espaços."""
    if not isinstance(texto, str):
        return ""
    t = unicodedata.normalize("NFKD", texto)
    t = "".join(ch for ch in t if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", t).strip().lower()


def achar_linha_cabecalho(caminho, limite=15):
    """Devolve o índice da linha que contém o cabeçalho real das medições."""
    with open(caminho, "r", encoding="latin1", errors="ignore") as fh:
        for i, linha in enumerate(fh):
            if i >= limite:
                break
            n = normalizar(linha)
            if "data" in n and ("hora" in n or "precipitacao" in n):
                return i
    return 8  # fallback: comportamento antigo


def renomear_colunas(df):
    """Traduz os nomes do INMET para as chaves internas, sem colisão."""
    novo, ja_usados = {}, set()

    for pos, col in enumerate(df.columns):
        n = normalizar(col)

        # Data e hora: identificadas por posição/nome exato, nunca por fragmento.
        if pos == 0 or n == "data" or n.startswith("data "):
            novo[col] = "data"
            continue
        if pos == 1 or n in ("hora", "hora utc") or n.startswith("hora utc"):
            novo[col] = "hora"
            continue

        for fragmento, interno in MAPA_COLUNAS:
            if fragmento in n and interno not in ja_usados:
                novo[col] = interno
                ja_usados.add(interno)
                break

    return df.rename(columns=novo)


def converter_datas(serie):
    """INMET usa AAAA/MM/DD em alguns anos e DD/MM/AAAA em outros.

    Testar os dois formatos explicitamente evita o erro silencioso de
    `dayfirst=True`, que lê 2023/01/05 como 5 de janeiro em um arquivo e
    como 1 de maio em outro.
    """
    amostra = serie.dropna().astype(str).head(50)
    if amostra.empty:
        return pd.to_datetime(serie, errors="coerce")

    for formato in ("%Y/%m/%d", "%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        convertido = pd.to_datetime(serie, format=formato, errors="coerce")
        if convertido.notna().mean() > 0.95:
            return convertido

    return pd.to_datetime(serie, errors="coerce", dayfirst=True)


def ler_arquivo_inmet(caminho):
    """Lê um CSV horário do INMET já limpo, ou devolve None se não der."""
    linha_cab = achar_linha_cabecalho(caminho)

    df = pd.read_csv(
        caminho, sep=";", skiprows=linha_cab, encoding="latin1",
        decimal=",", na_values=SENTINELAS, keep_default_na=True,
    )

    # Colunas fantasma nascem do ";" no fim de cada linha.
    df = df.loc[:, ~df.columns.str.contains(r"^Unnamed", na=False)]
    df = df.dropna(axis=1, how="all")

    df = renomear_colunas(df)

    if "data" not in df.columns:
        return None

    df["data"] = converter_datas(df["data"])
    df = df.dropna(subset=["data"])

    # Tudo que é medição precisa ser número.
    for col in LIMITES_FISICOS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Sentinelas que escaparam + valores fisicamente impossíveis viram nulo.
    for col, (minimo, maximo) in LIMITES_FISICOS.items():
        if col in df.columns:
            df.loc[df[col] <= -900, col] = np.nan
            df.loc[(df[col] < minimo) | (df[col] > maximo), col] = np.nan

    return df


def extrair_metadados(nome_arquivo):
    """INMET_CO_MT_A901_CUIABA_01-01-2023_A_31-12-2023.CSV -> (MT, A901, CUIABA)"""
    base = os.path.basename(nome_arquivo).upper().replace(".CSV", "")
    partes = base.split("_")
    if len(partes) < 5:
        return None
    uf = partes[2]
    cod_estacao = partes[3]
    # A cidade pode ter "_" no nome; junta tudo até a primeira data.
    resto = partes[4:]
    pedacos = []
    for p in resto:
        if re.match(r"^\d{2}-\d{2}-\d{4}$", p) or p == "A":
            break
        pedacos.append(p)
    cidade = " ".join(pedacos).strip()
    return uf, cod_estacao, limpar_nome_cidade(cidade)


def limpar_nome_cidade(nome):
    """Padroniza o nome da cidade para o merge não falhar por espaço/hífen."""
    n = nome.upper().strip()
    n = re.sub(r"\s*-\s*", " - ", n)   # "PORTO ALEGRE- BELEM" -> "PORTO ALEGRE - BELEM"
    n = re.sub(r"\s+", " ", n)          # "SAO FELIX  DO" -> "SAO FELIX DO"
    return n


print("Funções de leitura carregadas.")

## 3. Agregação horária → diária

Cada variável tem a regra que faz sentido para ela. Foi aqui que a versão anterior
errou ao aplicar `.mean()` em tudo:

- **Precipitação e radiação são fluxos acumulados** — a pergunta agronômica é *quanto choveu
  no mês*, não *qual a média de chuva por hora*. Somar também resolve a inconsistência entre
  estações que gravam `0` à noite e estações que deixam em branco: nos dois casos a soma dá
  o mesmo total.
- **Direção do vento é circular** — a média aritmética de 350° e 10° dá 180°, exatamente a
  direção oposta à real. Precisa de média vetorial.
- **Rajada máxima é um extremo** — a média de rajadas descaracteriza o evento; o que importa
  é a maior.

In [ ]:
def media_direcao_circular(series):
    """Média vetorial de ângulos em grau. Devolve 0-360, ou NaN se não houver dado."""
    s = series.dropna()
    if len(s) == 0:
        return np.nan
    rad = np.radians(s.values)
    seno, cosseno = np.sin(rad).mean(), np.cos(rad).mean()
    if np.hypot(seno, cosseno) < 1e-9:   # vetores se anulam: sem direção dominante
        return np.nan
    return float(np.degrees(np.arctan2(seno, cosseno)) % 360)


def consistencia_direcao(series):
    """0 = vento gira o tempo todo; 1 = sopra sempre da mesma direção."""
    s = series.dropna()
    if len(s) == 0:
        return np.nan
    rad = np.radians(s.values)
    return float(np.hypot(np.sin(rad).mean(), np.cos(rad).mean()))


def pressao_vapor_saturacao(temp_c):
    """Tetens: pressão de vapor de saturação em kPa."""
    return 0.6108 * np.exp(17.27 * temp_c / (temp_c + 237.3))


def agregar_para_diario(df):
    """Horário -> diário, cada variável com sua regra."""
    df = df.copy()
    df["dia"] = df["data"].dt.normalize()

    def col(nome):
        return df[nome] if nome in df.columns else pd.Series(np.nan, index=df.index)

    g = df.groupby("dia")
    diario = pd.DataFrame(index=g.size().index)

    # Quantas horas do dia realmente têm medição de temperatura.
    diario["horas_validas"] = col("temp_ar").groupby(df["dia"]).count()

    # --- acumulados: SOMA ---
    diario["precipitacao_mm"] = col("precipitacao").groupby(df["dia"]).sum(min_count=1)
    diario["radiacao_kjm2"]   = col("radiacao").groupby(df["dia"]).sum(min_count=1)

    # --- estados instantâneos: MÉDIA ---
    diario["temp_media"]    = col("temp_ar").groupby(df["dia"]).mean()
    diario["umidade_media"] = col("umidade").groupby(df["dia"]).mean()
    diario["pressao_media"] = col("pressao").groupby(df["dia"]).mean()
    diario["orvalho_medio"] = col("orvalho").groupby(df["dia"]).mean()
    diario["vento_medio"]   = col("vento_vel").groupby(df["dia"]).mean()

    # --- extremos do dia: MAX / MIN ---
    # Usa a coluna de máx/mín do INMET quando existe; senão cai na horária.
    tmax_fonte = "temp_max" if "temp_max" in df.columns else "temp_ar"
    tmin_fonte = "temp_min" if "temp_min" in df.columns else "temp_ar"
    diario["temp_max"] = col(tmax_fonte).groupby(df["dia"]).max()
    diario["temp_min"] = col(tmin_fonte).groupby(df["dia"]).min()

    umin_fonte = "umidade_min" if "umidade_min" in df.columns else "umidade"
    diario["umidade_min"]   = col(umin_fonte).groupby(df["dia"]).min()
    diario["vento_rajada"]  = col("vento_rajada").groupby(df["dia"]).max()

    # --- circular: decompõe em seno/cosseno, faz a média, recompõe o ângulo ---
    if "vento_dir" in df.columns:
        rad = np.radians(df["vento_dir"])
        componentes = pd.DataFrame({"dia": df["dia"],
                                    "sen": np.sin(rad),
                                    "cos": np.cos(rad)}).groupby("dia").mean()
        resultante = np.hypot(componentes["sen"], componentes["cos"])
        angulo = np.degrees(np.arctan2(componentes["sen"], componentes["cos"])) % 360
        diario["vento_dir"] = angulo.where(resultante >= 1e-9)
        diario["vento_dir_cons"] = resultante
    else:
        diario["vento_dir"] = np.nan
        diario["vento_dir_cons"] = np.nan

    # --- derivadas agronômicas ---
    # Graus-dia: energia térmica acumulada que a planta efetivamente usa.
    media_termica = (diario["temp_max"] + diario["temp_min"]) / 2
    diario["graus_dia"] = (media_termica - TEMP_BASE_GDD).clip(lower=0)

    # VPD: déficit de pressão de vapor, o indicador de demanda hídrica da atmosfera.
    es = pressao_vapor_saturacao(diario["temp_media"])
    diario["vpd_kpa"] = (es * (1 - diario["umidade_media"] / 100)).clip(lower=0)

    diario["amplitude_termica"] = diario["temp_max"] - diario["temp_min"]

    # Dia incompleto não vira média confiável.
    diario.loc[diario["horas_validas"] < MIN_HORAS_DIA,
               diario.columns.difference(["horas_validas"])] = np.nan

    return diario.reset_index()


print("Agregação diária definida.")

## 4. Agregação diária → mensal

O mês é a unidade de entrada do modelo. Além das estatísticas usuais, saem daqui duas
variáveis que só existem no nível diário e pesam muito em produtividade:

- **`maior_veranico`** — a maior sequência de dias seguidos sem chuva. Trinta dias somando
  120 mm bem distribuídos e 120 mm em dois temporais são lavouras completamente diferentes,
  e a média mensal não distingue as duas.
- **`graus_dia_acum`** — soma dos graus-dia do mês, a medida padrão de energia térmica
  acumulada usada em estimativa de ciclo e produtividade.

In [ ]:
def maior_sequencia_seca(chuva_diaria, limiar=LIMIAR_DIA_CHUVOSO_MM):
    """Maior nº de dias consecutivos com chuva abaixo do limiar."""
    seco = (chuva_diaria.fillna(0) < limiar).astype(int)
    maior = atual = 0
    for v in seco:
        atual = atual + 1 if v else 0
        maior = max(maior, atual)
    return int(maior)


def agregar_para_mensal(diario):
    """Diário -> mensal."""
    d = diario.copy()
    d["Ano_Mes"] = d["dia"].dt.strftime("%Y%m")
    linhas = []

    for ano_mes, bloco in d.groupby("Ano_Mes"):
        dias_no_mes = pd.Period(ano_mes, freq="M").days_in_month
        dias_validos = int(bloco["temp_media"].notna().sum())

        linhas.append({
            "Ano_Mes": ano_mes,
            "dias_validos": dias_validos,
            "dias_no_mes": dias_no_mes,
            "cobertura": round(dias_validos / dias_no_mes, 3),

            # temperatura
            "temp_media":        bloco["temp_media"].mean(),
            "temp_max_media":    bloco["temp_max"].mean(),
            "temp_min_media":    bloco["temp_min"].mean(),
            "temp_max_abs":      bloco["temp_max"].max(),
            "temp_min_abs":      bloco["temp_min"].min(),
            "amplitude_termica": bloco["amplitude_termica"].mean(),
            "graus_dia_acum":    bloco["graus_dia"].sum(min_count=1),

            # água
            "precipitacao_total_mm": bloco["precipitacao_mm"].sum(min_count=1),
            "dias_com_chuva":        int((bloco["precipitacao_mm"] >= LIMIAR_DIA_CHUVOSO_MM).sum()),
            "maior_veranico":        maior_sequencia_seca(bloco["precipitacao_mm"]),
            "umidade_media":         bloco["umidade_media"].mean(),
            "umidade_min_abs":       bloco["umidade_min"].min(),
            "vpd_medio_kpa":         bloco["vpd_kpa"].mean(),
            "orvalho_medio":         bloco["orvalho_medio"].mean(),

            # energia
            "radiacao_total_mjm2": bloco["radiacao_kjm2"].sum(min_count=1) / 1000.0,

            # atmosfera / vento
            "pressao_media":       bloco["pressao_media"].mean(),
            "vento_medio":         bloco["vento_medio"].mean(),
            "vento_rajada_max":    bloco["vento_rajada"].max(),
            "vento_dir_predom":    media_direcao_circular(bloco["vento_dir"]),
            "vento_dir_consistencia": bloco["vento_dir_cons"].mean(),
        })

    mensal = pd.DataFrame(linhas)

    # Mês com poucos dias não representa o mês.
    incompleto = mensal["cobertura"] < MIN_FRACAO_DIAS_MES
    colunas_medida = mensal.columns.difference(
        ["Ano_Mes", "dias_validos", "dias_no_mes", "cobertura"])
    mensal.loc[incompleto, colunas_medida] = np.nan

    return mensal


print("Agregação mensal definida.")

## 5. Processamento de todos os arquivos

In [ ]:
def processar_tudo(caminho_base):
    blocos, falhas = [], []

    padrao = os.path.join(caminho_base, "ano_*", "*.CSV")
    arquivos = sorted(glob.glob(padrao)) + sorted(glob.glob(padrao.replace(".CSV", ".csv")))
    arquivos = sorted(set(arquivos))

    for caminho in arquivos:
        pasta = os.path.basename(os.path.dirname(caminho))
        ano_pasta = pasta.split("_")[-1]
        if not (ano_pasta.isdigit() and int(ano_pasta) >= ANO_INICIAL):
            continue

        meta = extrair_metadados(caminho)
        if meta is None:
            falhas.append((os.path.basename(caminho), "nome fora do padrão"))
            continue
        uf, cod_estacao, cidade = meta
        if uf not in ESTADOS_ALVO:
            continue

        try:
            bruto = ler_arquivo_inmet(caminho)
            if bruto is None or bruto.empty:
                falhas.append((os.path.basename(caminho), "sem linhas úteis"))
                continue

            mensal = agregar_para_mensal(agregar_para_diario(bruto))
            mensal.insert(1, "UF", uf)
            mensal.insert(2, "Cidade", cidade)
            mensal.insert(3, "Cod_Estacao", cod_estacao)
            blocos.append(mensal)

        except Exception as erro:
            falhas.append((os.path.basename(caminho), str(erro)[:80]))

    if not blocos:
        raise RuntimeError("Nenhum arquivo processado — confira CAMINHO_BASE.")

    df = pd.concat(blocos, ignore_index=True)

    # Uma estação pode ter arquivos separados por ano: consolida.
    df = df.sort_values(["Cod_Estacao", "Ano_Mes"]).reset_index(drop=True)

    print(f"Arquivos processados: {len(blocos)} | falhas: {len(falhas)}")
    if falhas:
        print("\nPrimeiras falhas:")
        for nome, motivo in falhas[:10]:
            print(f"  - {nome}: {motivo}")
    return df, falhas


df_clima, falhas = processar_tudo(CAMINHO_BASE)

print(f"\nBase climática: {df_clima.shape[0]} linhas x {df_clima.shape[1]} colunas")
print(f"Estações: {df_clima['Cod_Estacao'].nunique()} | meses: {df_clima['Ano_Mes'].nunique()}")
df_clima.head()

## 6. Base de solos

Duas mudanças em relação à versão anterior.

**A tabela de referência ganhou atributos numéricos.** Texto (`'Argilosa'`, a descrição em
prosa) não entra em regressão. As colunas novas — capacidade de retenção de água, drenagem,
aptidão para grãos, profundidade — são ordinais de 1 a 5 e podem ir direto para o modelo,
que é o que dá sentido a um coeficiente de solo.

**O mapa cidade → solo virou uma tabela auditável.** A versão anterior tentou mapear por
cidade, o merge devolveu `NaN` em tudo (Cuiabá não estava na lista) e o erro passou
despercebido; o passo seguinte então aplicou uma regra por estado inteiro, que colapsa 180
estações em 2 classes. Aqui o mapa por cidade tem uma coluna `fonte`, o fallback por estado
é explícito, e no fim sai um **relatório de cobertura** dizendo quantas estações usam dado
real e quantas usam fallback — o merge não pode mais falhar em silêncio.

> ⚠️ **Preencher antes de defender o TCC.** As linhas marcadas como `fallback_uf` são
> aproximação, não levantamento pedológico. A fonte correta é o *Mapa de Solos do Brasil*
> (EMBRAPA/IBGE, escala 1:5.000.000) ou o SiBCS por município. O próprio TCC define o
> agrônomo como stakeholder responsável pela validação desses parâmetros — este é o ponto
> exato onde essa validação entra.

In [ ]:
# ------------------------------------------------ tabela de referência --
df_solo = pd.DataFrame({
    "id_solo": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "nome_solo": [
        "Latossolo Vermelho", "Latossolo Amarelo", "Argissolo Vermelho-Amarelo",
        "Neossolo Quartzarênico", "Cambissolo Háplico", "Vertissolo Háplico",
        "Neossolo Litólico", "Gleissolo Háplico", "Espodossolo Ferrocárbico",
        "Plintossolo Argilúvico",
    ],
    "ordem_solo": [
        "Latossolo", "Latossolo", "Argissolo", "Neossolo", "Cambissolo",
        "Vertissolo", "Neossolo", "Gleissolo", "Espodossolo", "Plintossolo",
    ],
    "textura": [
        "Argilosa", "Média/Argilosa", "Média/Argilosa", "Arenosa", "Diferenciada",
        "Muito Argilosa", "Rasa/Diferenciada", "Argilosa/Siltosa", "Arenosa",
        "Média/Argilosa",
    ],
    # --- ordinais 1-5: é isto que o Elastic Net consegue usar ---
    "retencao_agua":    [4, 4, 3, 1, 3, 5, 1, 5, 1, 3],
    "drenagem":         [5, 5, 4, 5, 3, 2, 4, 1, 5, 2],
    "aptidao_graos":    [5, 4, 4, 2, 2, 3, 1, 2, 1, 2],
    "profundidade":     [5, 5, 4, 4, 2, 3, 1, 3, 4, 3],
    "risco_erosao":     [1, 1, 4, 3, 4, 3, 5, 1, 2, 3],
    "culturas_predominantes": [
        "Soja, Milho, Trigo", "Soja, Milho", "Trigo, Milho, Pastagem",
        "Soja (com irrigação), Pastagem", "Trigo, Pastagem", "Arroz irrigado, Soja",
        "Pastagem nativa", "Arroz irrigado", "Pastagem", "Pastagem, Arroz",
    ],
    "descricao": [
        "Solo profundo e bem drenado. Excelente retenção de nutrientes, ideal para grãos.",
        "Solo profundo, exige correção com calcário para alta produtividade de grãos.",
        "Gradiente textural. Bom para trigo no Sul, mas exige cuidado com erosão.",
        "Arenoso e muito drenado. Exige irrigação e manejo de precisão para grãos.",
        "Raso e acidentado. Uso restrito para agricultura mecanizada.",
        "Argiloso, racha no seco e alaga na chuva. Exige manejo específico.",
        "Muito raso, com rochas. Inadequado para agricultura intensiva de grãos.",
        "Hidromórfico, frequentemente alagado. Condição ideal para arroz irrigado.",
        "Arenoso e pobre em nutrientes. Baixa aptidão para grãos sem alto investimento.",
        "Restrição temporária de drenagem. Uso limitado para culturas de sequeiro.",
    ],
})

# Escalas ordinais: 1 = baixo, 5 = alto (risco_erosao: 5 = mais crítico).
ESCALAS_ORDINAIS = ["retencao_agua", "drenagem", "aptidao_graos",
                    "profundidade", "risco_erosao"]

print(f"Tabela de solos: {len(df_solo)} classes, "
      f"{len(ESCALAS_ORDINAIS)} atributos numéricos.")
df_solo[["id_solo", "nome_solo"] + ESCALAS_ORDINAIS]

In [ ]:
# ------------------------------- mapa auditável cidade -> solo ---------
# Preencher com o Mapa de Solos EMBRAPA/IBGE. Cada linha declara sua origem.
# fonte = "levantamento"  -> conferido em fonte pedológica
# fonte = "regional"      -> generalização da região agrícola
MAPA_CIDADE_SOLO = pd.DataFrame([
    # --- RS: várzeas de arroz irrigado (Gleissolo), não Argissolo ---
    {"UF": "RS", "Cidade": "URUGUAIANA",               "id_solo": 8, "fonte": "regional"},
    {"UF": "RS", "Cidade": "SANTA VITORIA DO PALMAR",  "id_solo": 8, "fonte": "regional"},
    {"UF": "RS", "Cidade": "ALEGRETE",                 "id_solo": 8, "fonte": "regional"},
    # --- RS: planalto de grãos (Latossolo Vermelho) ---
    {"UF": "RS", "Cidade": "PASSO FUNDO",              "id_solo": 1, "fonte": "regional"},
    {"UF": "RS", "Cidade": "CRUZ ALTA",                "id_solo": 1, "fonte": "regional"},
    # --- MT: Latossolo do Cerrado ---
    {"UF": "MT", "Cidade": "SORRISO",                  "id_solo": 1, "fonte": "regional"},
    {"UF": "MT", "Cidade": "CAMPO NOVO DOS PARECIS",   "id_solo": 1, "fonte": "regional"},
    # --- PR / SP: Latossolo Vermelho ---
    {"UF": "PR", "Cidade": "CASCAVEL",                 "id_solo": 1, "fonte": "regional"},
    {"UF": "SP", "Cidade": "ITUVERAVA",                "id_solo": 1, "fonte": "regional"},
])

# Normaliza a chave dos dois lados — foi isto que quebrou o merge antes.
MAPA_CIDADE_SOLO["Cidade"] = MAPA_CIDADE_SOLO["Cidade"].map(limpar_nome_cidade)

# Fallback por estado, aplicado só onde não há linha específica.
FALLBACK_UF = {"SP": 1, "PR": 1, "MT": 1, "RS": 3}


def aplicar_solo(df_clima, mapa, fallback):
    df = df_clima.copy()
    df["Cidade"] = df["Cidade"].map(limpar_nome_cidade)

    df = df.merge(mapa[["UF", "Cidade", "id_solo", "fonte"]],
                  on=["UF", "Cidade"], how="left")

    sem_mapa = df["id_solo"].isna()
    df.loc[sem_mapa, "id_solo"] = df.loc[sem_mapa, "UF"].map(fallback)
    df.loc[sem_mapa, "fonte"] = "fallback_uf"

    orfaos = df["id_solo"].isna().sum()
    if orfaos:
        print(f"⚠️  {orfaos} linhas sem solo (UF fora do fallback).")

    df["id_solo"] = df["id_solo"].astype("Int64")
    return df


df_com_solo = aplicar_solo(df_clima, MAPA_CIDADE_SOLO, FALLBACK_UF)

# ---------------------------- relatório de cobertura (o merge não falha calado) --
cobertura = (df_com_solo.drop_duplicates("Cod_Estacao")
             .groupby("fonte").size().rename("estacoes").reset_index())
cobertura["%"] = (100 * cobertura["estacoes"]
                  / cobertura["estacoes"].sum()).round(1)

print("\nCOBERTURA DO MAPEAMENTO DE SOLO (por estação)")
print(cobertura.to_string(index=False))
pct_real = cobertura.loc[cobertura["fonte"] != "fallback_uf", "%"].sum()
print(f"\n{pct_real:.1f}% das estações têm solo mapeado; "
      f"o restante usa fallback por estado e precisa de validação agronômica.")

In [ ]:
# Merge final com a tabela de referência
df_tcc_completo = df_com_solo.merge(
    df_solo.drop(columns=["descricao"]),   # descrição é texto longo, fica fora do modelo
    on="id_solo", how="left",
)

# Ano_Mes como inteiro (202301) para ordenar e cruzar sem ambiguidade
df_tcc_completo["Ano_Mes"] = df_tcc_completo["Ano_Mes"].astype(int)

# Ordenação amigável das colunas
IDENT   = ["Ano_Mes", "UF", "Cidade", "Cod_Estacao"]
QUALID  = ["dias_validos", "dias_no_mes", "cobertura"]
SOLO    = ["id_solo", "nome_solo", "ordem_solo", "textura", "fonte"] + \
          ESCALAS_ORDINAIS + ["culturas_predominantes"]
CLIMA   = [c for c in df_tcc_completo.columns
           if c not in IDENT + QUALID + SOLO]

df_tcc_completo = df_tcc_completo[IDENT + QUALID + CLIMA + SOLO]
df_tcc_completo = df_tcc_completo.round(2)

print(f"Base final: {df_tcc_completo.shape[0]} linhas x {df_tcc_completo.shape[1]} colunas")
df_tcc_completo.head()

## 7. Validação da base

Esta célula é a rede de proteção: ela responde, com número, se a base pode ir para o
modelo. Roda tudo em cima do resultado — não confia em nenhuma etapa anterior.

In [ ]:
def validar(df):
    print("=" * 62)
    print("RELATÓRIO DE VALIDAÇÃO")
    print("=" * 62)

    print(f"\nLinhas: {len(df)} | Estações: {df['Cod_Estacao'].nunique()} "
          f"| Meses: {df['Ano_Mes'].nunique()}")
    print(f"Período: {df['Ano_Mes'].min()} a {df['Ano_Mes'].max()}")

    # --- 1. duplicatas: uma estação não pode ter dois registros do mesmo mês ---
    dup = df.duplicated(subset=["Cod_Estacao", "Ano_Mes"]).sum()
    print(f"\n[1] Duplicatas (estação+mês): {dup}  {'OK' if dup == 0 else 'CORRIGIR'}")

    # --- 2. completude ---
    print("\n[2] Completude por coluna (piores 8):")
    comp = (100 * df.notna().sum() / len(df)).sort_values()
    for nome, pct in comp.head(8).items():
        alerta = "  <-- abaixo de 85%" if pct < 85 else ""
        print(f"     {nome:26s} {pct:5.1f}%{alerta}")

    # --- 3. faixas físicas ---
    print("\n[3] Faixas físicas:")
    checagens = {
        "temp_media":            (-10, 45),
        "umidade_media":         (0, 100),
        "precipitacao_total_mm": (0, 1500),
        "vento_dir_predom":      (0, 360),
        "pressao_media":         (800, 1060),
        "radiacao_total_mjm2":   (0, 1200),
    }
    for col, (lo, hi) in checagens.items():
        if col not in df.columns:
            continue
        fora = ((df[col] < lo) | (df[col] > hi)).sum()
        print(f"     {col:26s} fora de [{lo}, {hi}]: {fora}  "
              f"{'OK' if fora == 0 else 'VERIFICAR'}")

    # --- 4. coerência interna ---
    print("\n[4] Coerência interna:")
    inc_t = (df["temp_max_abs"] < df["temp_min_abs"]).sum()
    print(f"     temp_max_abs < temp_min_abs: {inc_t}  {'OK' if inc_t == 0 else 'CORRIGIR'}")
    inc_d = (df["dias_com_chuva"] > df["dias_no_mes"]).sum()
    print(f"     dias_com_chuva > dias_no_mes: {inc_d}  {'OK' if inc_d == 0 else 'CORRIGIR'}")

    # --- 5. direção do vento: o teste que denuncia média aritmética ---
    if "vento_dir_predom" in df.columns:
        d = df["vento_dir_predom"].dropna()
        if len(d):
            pct_meio = 100 * ((d >= 135) & (d <= 225)).mean()
            print(f"\n[5] Direção do vento entre 135° e 225°: {pct_meio:.0f}%")
            print("     (acima de ~60% indica média aritmética em variável circular)")

    # --- 6. colinearidade que sobrou ---
    num = df.select_dtypes(include=[np.number]).drop(
        columns=["Ano_Mes", "id_solo", "dias_no_mes"], errors="ignore")
    corr = num.corr().abs()
    altos = [(corr.columns[i], corr.columns[j], corr.iloc[i, j])
             for i in range(len(corr)) for j in range(i + 1, len(corr))
             if corr.iloc[i, j] > 0.90]
    print(f"\n[6] Pares com |r| > 0,90: {len(altos)}")
    for a, b, v in sorted(altos, key=lambda t: -t[2])[:8]:
        print(f"     {a} x {b}: {v:.3f}")
    if not altos:
        print("     nenhum — redundância resolvida na origem")

    # --- 7. solo ---
    print(f"\n[7] Classes de solo em uso: {df['nome_solo'].nunique()}")
    print(df.drop_duplicates("Cod_Estacao")["fonte"].value_counts().to_string())

    print("\n" + "=" * 62)


validar(df_tcc_completo)

## 8. Exportação

In [ ]:
df_tcc_completo.to_excel(SAIDA_XLSX, index=False)
print(f"Salvo: {SAIDA_XLSX}")
print(f"{df_tcc_completo.shape[0]} linhas x {df_tcc_completo.shape[1]} colunas")

# Dicionário de dados — vai direto para o capítulo de modelagem do TCC
DICIONARIO = {
    "Ano_Mes": "Ano e mês da observação (AAAAMM)",
    "UF": "Unidade federativa da estação",
    "Cidade": "Município da estação (nome normalizado)",
    "Cod_Estacao": "Código da estação automática do INMET",
    "dias_validos": "Dias do mês com pelo menos 18 h de medição",
    "dias_no_mes": "Total de dias do mês (referência de cobertura)",
    "cobertura": "dias_validos / dias_no_mes",
    "temp_media": "Temperatura média do ar no mês (°C)",
    "temp_max_media": "Média das máximas diárias (°C)",
    "temp_min_media": "Média das mínimas diárias (°C)",
    "temp_max_abs": "Maior temperatura registrada no mês (°C)",
    "temp_min_abs": "Menor temperatura registrada no mês (°C)",
    "amplitude_termica": "Média diária de (máxima - mínima) (°C)",
    "graus_dia_acum": f"Graus-dia acumulados, base {TEMP_BASE_GDD} °C",
    "precipitacao_total_mm": "Chuva acumulada no mês (mm)",
    "dias_com_chuva": f"Dias com chuva >= {LIMIAR_DIA_CHUVOSO_MM} mm",
    "maior_veranico": "Maior sequência de dias consecutivos sem chuva",
    "umidade_media": "Umidade relativa média (%)",
    "umidade_min_abs": "Menor umidade relativa do mês (%)",
    "vpd_medio_kpa": "Déficit de pressão de vapor médio (kPa)",
    "orvalho_medio": "Temperatura média do ponto de orvalho (°C)",
    "radiacao_total_mjm2": "Radiação global acumulada no mês (MJ/m²)",
    "pressao_media": "Pressão atmosférica média na estação (mB)",
    "vento_medio": "Velocidade média do vento (m/s)",
    "vento_rajada_max": "Maior rajada registrada no mês (m/s)",
    "vento_dir_predom": "Direção predominante do vento (°, média vetorial)",
    "vento_dir_consistencia": "0 = vento gira; 1 = direção constante",
    "id_solo": "Chave da classe de solo",
    "nome_solo": "Classe de solo (SiBCS)",
    "ordem_solo": "Ordem pedológica",
    "textura": "Classe textural",
    "fonte": "Origem do solo: regional, levantamento ou fallback_uf",
    "retencao_agua": "Capacidade de retenção hídrica (1-5)",
    "drenagem": "Capacidade de drenagem (1-5)",
    "aptidao_graos": "Aptidão para grãos (1-5)",
    "profundidade": "Profundidade efetiva (1-5)",
    "risco_erosao": "Suscetibilidade à erosão (1-5)",
    "culturas_predominantes": "Culturas típicas da classe de solo",
}

df_dicionario = pd.DataFrame(
    [{"coluna": k, "descricao": v, "presente": k in df_tcc_completo.columns}
     for k, v in DICIONARIO.items()])

with pd.ExcelWriter(SAIDA_XLSX, engine="openpyxl") as writer:
    df_tcc_completo.to_excel(writer, sheet_name="dados", index=False)
    df_dicionario.to_excel(writer, sheet_name="dicionario", index=False)
    df_solo.to_excel(writer, sheet_name="ref_solos", index=False)

print(f"\nArquivo final com 3 abas: dados, dicionario, ref_solos")

## 9. O que ainda falta para treinar o Elastic Net

Esta base é o conjunto de **entrada** (X). Para treinar o modelo ainda faltam duas peças,
e nenhuma delas está na planilha do INMET:

1. **A variável-alvo (y)** — produtividade em sacas/ha ou rentabilidade, por município e safra.
   Fontes: **PAM/IBGE** (Produção Agrícola Municipal, por município/ano/cultura) ou
   **CONAB** (levantamentos de safra). O cruzamento é por `Cidade` + `UF` + ano.
2. **A cultura efetivamente plantada** em cada registro — hoje `culturas_predominantes`
   descreve o que *costuma* ser plantado naquele solo, não o que foi plantado ali naquela safra.

Além disso, ao montar o `X` para o modelo:

- Agregar os meses pela **janela do ciclo da cultura** (soja em MT: out–fev), não pelo ano civil.
  O modelo prevê safra, e a safra não respeita o calendário.
- Manter `cobertura` como filtro, não como preditor: descartar linhas abaixo do limite em vez
  de deixar o modelo aprender sobre média fraca.
- Aplicar o Min-Max **dentro do fold** da validação cruzada, ajustando o scaler só no treino.
  Ajustar no conjunto inteiro antes de dividir vaza informação do teste e infla o R².